In [1]:
import sys
!{sys.executable} -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!{sys.executable} -m pip install transformers gradio Pillow opencv-python

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached transformers-5.8.1-py3-none-any.whl.metadata (33 kB)
  Using cached gradio-6.14.0-py3-none-any.whl.metadata (17 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached huggingface_hub-1.15.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached hf_xet-1.5.0-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached fastapi-0.136.1-py3-none-any.whl.metadata (28 kB)
  Using cached gradio_client-2.5.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached hf_gradio-0.4.1-py3-none-any.whl.metadata (428 bytes

In [2]:
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPU             : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

PyTorch version : 2.11.0+cu130
CUDA available  : True
GPU             : NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
import gradio as gr
import torch
from PIL import Image, ImageDraw
from transformers import OwlViTProcessor, OwlViTForObjectDetection, OwlViTImageProcessor
from collections import defaultdict
import numpy as np

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load model
MODEL_ID = "google/owlvit-base-patch32"
processor       = OwlViTProcessor.from_pretrained(MODEL_ID)
image_processor = OwlViTImageProcessor.from_pretrained(MODEL_ID)  # ← NEW
model           = OwlViTForObjectDetection.from_pretrained(MODEL_ID).to(device)
model.eval()
print("Model loaded successfully!")

Using device: cuda


Loading weights:   0%|          | 0/412 [00:00<?, ?it/s]

Model loaded successfully!


In [4]:
# color palette helper
PALETTE = [
    (255, 56,  56),   # red
    (56,  255, 56),   # green
    (56,  56,  255),  # blue
    (255, 255, 56),   # yellow
    (255, 56,  255),  # magenta
    (56,  255, 255),  # cyan
]

def get_color(idx):
    return PALETTE[idx % len(PALETTE)]

In [5]:
# Dectection Function
def detect_objects(image: Image.Image, text_prompt: str, threshold: float):
    # Parse comma-separated labels into OWL-ViT query format
    raw_labels = [lbl.strip() for lbl in text_prompt.split(",") if lbl.strip()]
    queries    = [[f"a photo of a {lbl}" for lbl in raw_labels]]

    # Pre-process inputs
    inputs = processor(
        text=queries,
        images=image,
        return_tensors="pt"
    ).to(device)

    # Inference
    with torch.no_grad():
        outputs = model(**inputs)

    # Post-process: convert to [x1, y1, x2, y2] boxes
    target_sizes = torch.tensor([image.size[::-1]])
    results = image_processor.post_process_object_detection(
        outputs=outputs,
        target_sizes=target_sizes,
        threshold=threshold
    )[0]

    return results, raw_labels

    # ── Apply NMS to remove duplicate overlapping boxes ──────────
    boxes  = results["boxes"]
    scores = results["scores"]
    labels = results["labels"]

    if len(boxes) > 0:
        keep   = nms(boxes, scores, iou_threshold=0.3)  # ← filters overlaps
        results = {
            "boxes":  boxes[keep],
            "scores": scores[keep],
            "labels": labels[keep]
        }

    return results, raw_labels

In [6]:
# Annotation and counting function
def annotate_image(image: Image.Image, results: dict, raw_labels: list):
    draw   = ImageDraw.Draw(image)
    counts = defaultdict(int)

    boxes  = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    labels = results["labels"].cpu().numpy()

    for box, score, label_idx in zip(boxes, scores, labels):
        label_name = raw_labels[label_idx]
        counts[label_name] += 1
        color = get_color(label_idx)

        x1, y1, x2, y2 = box
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        tag = f"{label_name} {score:.2f}"
        draw.rectangle([x1, y1 - 16, x1 + len(tag) * 7, y1], fill=color)
        draw.text((x1 + 2, y1 - 15), tag, fill="white")

    # Build count summary
    if counts:
        lines = [f"{'Object':<20} {'Count':>5}", "-" * 27]
        for name, cnt in sorted(counts.items()):
            lines.append(f"{name:<20} {cnt:>5}")
        lines.append("-" * 27)
        lines.append(f"{'TOTAL':<20} {sum(counts.values()):>5}")
        summary = "\n".join(lines)
    else:
        summary = "No objects detected. Try lowering the threshold or revising your prompt."

    return image, summary

In [7]:
# gradio pipeline function
def run_pipeline(image, text_prompt, threshold):
    if image is None:
        return None, "⚠️ Please upload an image."
    if not text_prompt.strip():
        return None, "⚠️ Please enter at least one object label."

    pil_image          = Image.fromarray(image).convert("RGB")
    results, raw_labels = detect_objects(pil_image, text_prompt, threshold)
    annotated, summary  = annotate_image(pil_image, results, raw_labels)
    return np.array(annotated), summary

In [8]:
# Launch Gradio app
with gr.Blocks(title="Adaptive Object Detection — OWL-ViT") as demo:
    gr.Markdown("""
    # 🎥 Adaptive Object Detection for Surveillance
    **Model:** OWL-ViT (`google/owlvit-base-patch32`)  
    Type object categories to count, separated by commas.  
    *Examples:* `car, truck, bus` · `person, bicycle` · `dog, cat`
    """)

    with gr.Row():
        with gr.Column(scale=1):
            image_input      = gr.Image(label="Upload Surveillance Image")
            prompt_input     = gr.Textbox(
                label="Object Prompt (comma-separated)",
                placeholder="car, truck, bus, motorcycle",
                value="car, truck, bus"
            )
            threshold_slider = gr.Slider(
                minimum=0.05, maximum=0.95, value=0.1, step=0.05,
                label="Confidence Threshold"
            )
            detect_btn = gr.Button("🔍 Detect & Count", variant="primary")

        with gr.Column(scale=1):
            image_output   = gr.Image(label="Annotated Output")
            summary_output = gr.Textbox(label="Object Count Summary", lines=8)

    detect_btn.click(
        fn=run_pipeline,
        inputs=[image_input, prompt_input, threshold_slider],
        outputs=[image_output, summary_output]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
